# 04. Deep Learning GRU Network (h=10) with Smooth Target-Normalized LR Finder — 200 Users

This notebook trains a **Gated Recurrent Unit (GRU)** Deep Learning model with **History Depth $h=10$** ($L=11$ sequence window timesteps) on the 5G NR 3.0 GHz **200-User Dataset** (`99,589 samples`).

### Target Representation: Relative Coordinates to Serving BS
- **Target Variables:** $(\Delta x, \Delta y, \Delta z) = (x_{\text{UE}} - x_{\text{BS}}, y_{\text{UE}} - y_{\text{BS}}, z_{\text{UE}} - z_{\text{BS}})$.
- **Why Relative is Superior:** Physical RF signals (RSS, SINR, AoA) depend directly on the relative vector from the serving BS. Relative targets provide **BS origin invariance**, allowing model weights to generalize across any cell location.
- **Target Standardization:** Target coordinates $(\Delta x, \Delta y, \Delta z)$ are standardized $(\mu=0, \sigma=1)$ during training for smooth, stable loss curves.
- **Multi-User Trajectory Plots:** Generates trajectory tracking plots for 5 diverse unseen test users, saved in `OUT_DIR / trajectory_plots`.

In [ ]:
import sys, os, time, warnings, json, math, datetime, glob, copy
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
from scipy.linalg import inv
from sklearn.metrics import mean_absolute_error

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader

NOTEBOOK_DIR = Path(os.getcwd())
PROJECT_ROOT = NOTEBOOK_DIR
for _ in range(10):
    if (PROJECT_ROOT / 'results' / 'grid_localization' / 'grid_25x25').exists():
        break
    if PROJECT_ROOT.parent == PROJECT_ROOT:
        break
    PROJECT_ROOT = PROJECT_ROOT.parent

sys.path.insert(0, str(PROJECT_ROOT / 'experiments' / '09_grid_localization' / 'src' / 'python'))

from pipelines.multi_user_200_pipeline import load_200_users, make_unseen_user_split
from pipelines.multi_user_pipeline_regression import _read_bs_position_3d

RUN_TIMESTAMP = datetime.datetime.now().strftime('%Y-%m-%d_%H-%M-%S')
OUT_DIR = PROJECT_ROOT / 'results' / 'notebook_experiments' / 'multi_user_poc' / f'gru_h10_{RUN_TIMESTAMP}'
PLOT_DIR = OUT_DIR / 'trajectory_plots'
PLOT_DIR.mkdir(parents=True, exist_ok=True)
print(f'Output directory: {OUT_DIR}')
print(f'Trajectory plots directory: {PLOT_DIR}')

## Global Hyperparameters & Configurable Parameters (TOP-LEVEL)

In [ ]:
# ── Top-Level Configurable Parameters ───────────────────────────────────
SEED                = 42         # Random seed for reproducibility
TRAIN_USER_RATIO    = 0.80       # 80% Train Users (160) / 20% Unseen Test Users (40)
H_TARGET            = 10         # History depth (h=10 sequence window L=11)
SEQUENCE_LENGTH     = H_TARGET + 1 # L = 11 timesteps

# Training & Architecture Parameters
BATCH_SIZE          = 2048       # DataLoader batch size
EPOCHS              = 50         # Number of training epochs
LEARNING_RATE       = 1e-3       # Initial learning rate (auto-updated by LR Finder)
WEIGHT_DECAY        = 1e-4       # L2 Weight decay regularization
GRU_HIDDEN_DIM      = 128        # GRU hidden state dimension
GRU_NUM_LAYERS      = 2          # Number of stacked GRU layers
DROPOUT_RATE        = 0.20       # Dropout rate

# LR Finder Parameters (Extended Range to 1e-7)
LR_START            = 1e-7       # Lower starting LR for smooth baseline
LR_END              = 1e-1       # Ending LR for sweep
LR_STEPS            = 120        # Number of mini-batches for LR sweep

# Kinematic Post-Processing Parameters
PROCESS_NOISE_STD   = 0.5        # Kalman process noise std
R_STD               = 15.0       # Kalman measurement noise std

torch.manual_seed(SEED)
np.random.seed(SEED)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using Compute Device: {device}')
print(f'Configured GRU: h={H_TARGET} (L={SEQUENCE_LENGTH}) | LR Sweep: {LR_START:.1e} -> {LR_END:.1e}')

## Load 200-User Dataset & Apply 4° AoA Noise Model

In [ ]:
data_base = PROJECT_ROOT / 'results' / 'grid_localization' / 'grid_25x25'
dirs = sorted(list(data_base.glob('sim_data_200users_*')))
if not dirs:
    raise FileNotFoundError('No sim_data_200users_* dataset directories found!')
DATA_DIR = dirs[-1]
print(f'Loading 200-User dataset from: {DATA_DIR}')

df_raw = load_200_users(DATA_DIR)
print(f'Loaded {len(df_raw):,} records across {len(df_raw["user_id"].unique())} diverse mobile users.')

# Compute Relative Displacements (target_x, target_y, target_z) Relative to Serving BS
bs_pos = np.array(_read_bs_position_3d(DATA_DIR))
ue_xyz = np.column_stack([df_raw['x_pos'], df_raw['y_pos'], df_raw['ue_height']])
delta  = ue_xyz - bs_pos
df_raw['target_x'], df_raw['target_y'], df_raw['target_z'] = delta[:,0], delta[:,1], delta[:,2]
print(f'Serving Base Station Location: [{bs_pos[0]:.1f}, {bs_pos[1]:.1f}, {bs_pos[2]:.1f}] m')
print('Targets computed as relative displacement (UE_xyz - BS_xyz).')

AOA_NOISE_STD_DEG = 4.0
AOA_QUANT_STEP_DEG = 5.0
mask_ant = df_raw['n_antennas'] > 1
rng_noise = np.random.RandomState(SEED)
df_raw.loc[mask_ant, 'aoa_azimuth'] = df_raw.loc[mask_ant, 'aoa_azimuth'].values + (AOA_NOISE_STD_DEG * rng_noise.randn(mask_ant.sum())).astype(np.float32)
df_raw.loc[mask_ant, 'aoa_elevation'] = df_raw.loc[mask_ant, 'aoa_elevation'].values + (AOA_NOISE_STD_DEG * rng_noise.randn(mask_ant.sum())).astype(np.float32)
df_raw.loc[mask_ant, 'aoa_azimuth'] = np.round(df_raw.loc[mask_ant, 'aoa_azimuth'] / AOA_QUANT_STEP_DEG) * AOA_QUANT_STEP_DEG
df_raw.loc[mask_ant, 'aoa_elevation'] = np.round(df_raw.loc[mask_ant, 'aoa_elevation'] / AOA_QUANT_STEP_DEG) * AOA_QUANT_STEP_DEG
df_raw.loc[~mask_ant, ['aoa_azimuth', 'aoa_elevation']] = 0.0
print('Noise model applied.')

## Construct 3D Sequence Dataset & DataLoaders (Target Standardized)

In [ ]:
class SequenceCSIDataset(Dataset):
    def __init__(self, df, h=10, sig_mean=None, sig_std=None, stat_mean=None, stat_std=None, targ_mean=None, targ_std=None):
        self.samples = []
        signal_cols = ['rss', 'sinr', 'aoa_azimuth', 'aoa_elevation', 'delta_t']
        static_cols = ['n_antennas', 'antenna_gain_db', 'ue_height']
        target_cols = ['target_x', 'target_y', 'target_z']
        
        df = df.copy()
        self.sig_mean  = sig_mean  if sig_mean  is not None else df[signal_cols].mean().values
        self.sig_std   = sig_std   if sig_std   is not None else df[signal_cols].std().values + 1e-6
        self.stat_mean = stat_mean if stat_mean is not None else df[static_cols].mean().values
        self.stat_std  = stat_std  if stat_std  is not None else df[static_cols].std().values + 1e-6
        self.targ_mean = targ_mean if targ_mean is not None else df[target_cols].mean().values
        self.targ_std  = targ_std  if targ_std  is not None else df[target_cols].std().values + 1e-6
        
        df[signal_cols] = (df[signal_cols] - self.sig_mean) / self.sig_std
        df[static_cols] = (df[static_cols] - self.stat_mean) / self.stat_std
        df[target_cols] = (df[target_cols] - self.targ_mean) / self.targ_std
        
        L = h + 1
        for uid, udf in df.groupby('user_id'):
            udf = udf.sort_values('step_index').reset_index(drop=True)
            n_steps = len(udf)
            if n_steps < L:
                continue
            
            sigs  = udf[signal_cols].values.astype(np.float32)
            stats = udf[static_cols].values.astype(np.float32)
            targs = udf[target_cols].values.astype(np.float32)
            u_ids = udf['user_id'].values
            dts   = udf['delta_t'].values
            
            for i in range(h, n_steps):
                seq_sig  = sigs[i-h:i+1]
                stat_vec = stats[i]
                targ_vec = targs[i]
                self.samples.append({
                    'seq': seq_sig,
                    'static': stat_vec,
                    'target': targ_vec,
                    'user_id': u_ids[i],
                    'delta_t': dts[i]
                })
                
    def __len__(self):
        return len(self.samples)
        
    def __getitem__(self, idx):
        s = self.samples[idx]
        return {
            'seq': torch.tensor(s['seq'], dtype=torch.float32),
            'static': torch.tensor(s['static'], dtype=torch.float32),
            'target': torch.tensor(s['target'], dtype=torch.float32),
            'user_id': s['user_id'],
            'delta_t': s['delta_t']
        }

df_split, train_users, test_users = make_unseen_user_split(df_raw, train_ratio=TRAIN_USER_RATIO, seed=SEED)
df_train = df_split[df_split['split']=='train'].reset_index(drop=True)
df_test  = df_split[df_split['split']=='test'].reset_index(drop=True)

train_ds = SequenceCSIDataset(df_train, h=H_TARGET)
test_ds  = SequenceCSIDataset(df_test, h=H_TARGET,
                              sig_mean=train_ds.sig_mean, sig_std=train_ds.sig_std,
                              stat_mean=train_ds.stat_mean, stat_std=train_ds.stat_std,
                              targ_mean=train_ds.targ_mean, targ_std=train_ds.targ_std)

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, drop_last=False)
test_loader  = DataLoader(test_ds, batch_size=BATCH_SIZE, shuffle=False, drop_last=False)

print(f'Target Standardization Applied: mean={train_ds.targ_mean}, std={train_ds.targ_std}')
print(f'Dataset Built: {len(train_ds):,} Train Sequences ({len(train_users)} Users) | {len(test_ds):,} Test Sequences ({len(test_users)} Users)')

## Define GRU 3D Positioning Network

In [ ]:
class GRU3DPositioningNet(nn.Module):
    def __init__(self, input_dim=5, static_dim=3, hidden_dim=128, num_layers=2, dropout=0.20):
        super().__init__()
        self.gru = nn.GRU(input_size=input_dim, hidden_size=hidden_dim, num_layers=num_layers,
                          batch_first=True, dropout=dropout if num_layers > 1 else 0.0)
        
        fusion_dim = hidden_dim + static_dim
        self.mlp = nn.Sequential(
            nn.Linear(fusion_dim, 256),
            nn.LeakyReLU(0.1),
            nn.BatchNorm1d(256),
            nn.Dropout(dropout),
            nn.Linear(256, 128),
            nn.LeakyReLU(0.1),
            nn.Linear(128, 3)
        )
        
    def forward(self, x_seq, x_static):
        gru_out, _ = self.gru(x_seq)
        h_last = gru_out[:, -1, :]
        fused = torch.cat([h_last, x_static], dim=1)
        out = self.mlp(fused)
        return out

model = GRU3DPositioningNet(input_dim=5, static_dim=3, hidden_dim=GRU_HIDDEN_DIM, num_layers=GRU_NUM_LAYERS, dropout=DROPOUT_RATE).to(device)
print(model)

## Learning Rate Finder Sweep ($10^{-7} \to 10^{-1}$)

In [ ]:
# ── PyTorch Learning Rate Finder (1e-7 -> 1e-1) ──────────────────────────
def find_learning_rate(model, train_loader, lr_start=1e-7, lr_end=1e-1, num_steps=120, beta=0.98):
    init_state = copy.deepcopy(model.state_dict())
    model.train()
    
    mult = (lr_end / lr_start) ** (1.0 / num_steps)
    lr = lr_start
    optimizer = optim.AdamW(model.parameters(), lr=lr, weight_decay=WEIGHT_DECAY)
    criterion = nn.SmoothL1Loss()
    
    avg_loss = 0.0
    best_loss = float('inf')
    batch_num = 0
    
    lrs = []
    losses = []
    
    print(f'Starting Extended LR Finder Sweep from {lr_start:.1e} to {lr_end:.1e} over {num_steps} steps...')
    
    for batch in train_loader:
        batch_num += 1
        seq  = batch['seq'].to(device)
        stat = batch['static'].to(device)
        targ = batch['target'].to(device)
        
        optimizer.zero_grad()
        pred = model(seq, stat)
        loss = criterion(pred, targ)
        
        avg_loss = beta * avg_loss + (1 - beta) * loss.item()
        smoothed_loss = avg_loss / (1 - beta ** batch_num)
        
        if batch_num > 1 and smoothed_loss > 4.0 * best_loss:
            break
            
        if smoothed_loss < best_loss or batch_num == 1:
            best_loss = smoothed_loss
            
        lrs.append(lr)
        losses.append(smoothed_loss)
        
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()
        
        lr *= mult
        for param_group in optimizer.param_groups:
            param_group['lr'] = lr
            
        if batch_num >= num_steps:
            break
            
    model.load_state_dict(init_state)
    
    losses_arr = np.array(losses)
    grad = np.gradient(losses_arr)
    best_idx = np.argmin(grad)
    suggested_lr = lrs[best_idx]
    
    plt.figure(figsize=(9, 5))
    plt.plot(lrs, losses, 'b-', lw=2.5, label='Smoothed SmoothL1 Loss (Standardized Targets)')
    plt.axvline(suggested_lr, color='crimson', linestyle='--', lw=2, label=f'Suggested LR: {suggested_lr:.2e}')
    plt.xscale('log')
    plt.xlabel('Learning Rate (Log Scale)', fontsize=11)
    plt.ylabel('Normalized Loss (SmoothL1)', fontsize=11)
    plt.title('Smooth Learning Rate Finder Sweep ($10^{-7} \to 10^{-1}$)', fontsize=12, fontweight='bold')
    plt.grid(True, which='both', alpha=0.3)
    plt.legend(fontsize=10)
    plt.savefig(OUT_DIR / 'lr_finder_plot.png', dpi=150, bbox_inches='tight')
    plt.show()
    
    print(f'LR Finder Complete! Suggested Learning Rate: {suggested_lr:.2e}')
    return suggested_lr

suggested_lr = find_learning_rate(model, train_loader, lr_start=LR_START, lr_end=LR_END, num_steps=LR_STEPS)
LEARNING_RATE = suggested_lr

## Train GRU Network

In [ ]:
criterion = nn.SmoothL1Loss()
optimizer = optim.AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY)
scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS, eta_min=1e-5)

print(f'Starting GRU Training for {EPOCHS} Epochs with LR={LEARNING_RATE:.2e} on {device}...')
print('='*70)
t_start = time.time()

for epoch in range(1, EPOCHS + 1):
    model.train()
    tr_loss = 0.0
    for batch in train_loader:
        seq = batch['seq'].to(device)
        stat = batch['static'].to(device)
        targ = batch['target'].to(device)
        
        optimizer.zero_grad()
        pred = model(seq, stat)
        loss = criterion(pred, targ)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()
        tr_loss += loss.item() * len(targ)
        
    scheduler.step()
    tr_loss /= len(train_ds)
    
    if epoch % 5 == 0 or epoch == 1 or epoch == EPOCHS:
        model.eval()
        val_loss = 0.0
        val_errs = []
        with torch.no_grad():
            for batch in test_loader:
                seq = batch['seq'].to(device)
                stat = batch['static'].to(device)
                targ = batch['target'].to(device)
                pred = model(seq, stat)
                loss = criterion(pred, targ)
                val_loss += loss.item() * len(targ)
                
                pred_m = pred.cpu().numpy() * train_ds.targ_std + train_ds.targ_mean
                targ_m = targ.cpu().numpy() * train_ds.targ_std + train_ds.targ_mean
                errs = np.linalg.norm(pred_m - targ_m, axis=1)
                val_errs.extend(errs)
        val_loss /= len(test_ds)
        val_mae = np.mean(val_errs)
        print(f'Epoch {epoch:2d}/{EPOCHS:2d} | Train Loss: {tr_loss:.4f} | Val Loss: {val_loss:.4f} | Val 3D MAE: {val_mae:.3f}m | LR: {scheduler.get_last_lr()[0]:.6f}')

t_total = time.time() - t_start
print(f'Training Completed in {t_total:.2f} seconds!')

## Evaluate GRU Model on Unseen Test Users

In [ ]:
model.eval()
preds_list, targs_list, uids_list, dts_list = [], [], [], []
with torch.no_grad():
    for batch in test_loader:
        seq = batch['seq'].to(device)
        stat = batch['static'].to(device)
        targ = batch['target'].to(device)
        pred = model(seq, stat)
        
        pred_m = pred.cpu().numpy() * train_ds.targ_std + train_ds.targ_mean
        targ_m = targ.cpu().numpy() * train_ds.targ_std + train_ds.targ_mean
        
        preds_list.append(pred_m)
        targs_list.append(targ_m)
        uids_list.extend(batch['user_id'].numpy())
        dts_list.extend(batch['delta_t'].numpy())

preds_raw = np.vstack(preds_list)
targs = np.vstack(targs_list)
uids = np.array(uids_list)
dts  = np.array(dts_list)

errs_3d = np.linalg.norm(preds_raw - targs, axis=1)
mae_3d  = np.mean(errs_3d)
mae_x   = mean_absolute_error(targs[:, 0], preds_raw[:, 0])
mae_y   = mean_absolute_error(targs[:, 1], preds_raw[:, 1])
mae_z   = mean_absolute_error(targs[:, 2], preds_raw[:, 2])
p50     = np.percentile(errs_3d, 50)
p90     = np.percentile(errs_3d, 90)

print('=== GRU DEEP LEARNING BENCHMARK (UNSEEN 40 TEST USERS, h=10) ===')
print(f'3D Position MAE:                  {mae_3d:.3f} meters')
print(f'X-Axis MAE:                       {mae_x:.3f} meters')
print(f'Y-Axis MAE:                       {mae_y:.3f} meters')
print(f'Z-Axis MAE:                       {mae_z:.3f} meters')
print(f'50th Percentile (Median Error):   {p50:.3f} meters')
print(f'90th Percentile Error:           {p90:.3f} meters')

## Kinematic Post-Processing: Forward KF & RTS Smoother

In [ ]:
def run_kalman_and_rts(y_true, y_pred, delta_t_vec, process_noise_std=0.5, R_std=15.0):
    N = len(y_pred)
    if N == 0: return y_pred, y_pred
    H = np.zeros((3, 6)); H[0, 0], H[1, 1], H[2, 2] = 1.0, 1.0, 1.0
    R = (R_std ** 2) * np.eye(3)
    x_pred = np.zeros((N, 6)); P_pred = np.zeros((N, 6, 6))
    x_filt = np.zeros((N, 6)); P_filt = np.zeros((N, 6, 6))
    x_filt[0, :3] = y_pred[0]; P_filt[0] = np.eye(6) * 100.0
    x_pred[0], P_pred[0] = x_filt[0], P_filt[0]
    for t in range(1, N):
        dt = max(0.01, float(delta_t_vec[t]))
        F = np.eye(6); F[0, 3], F[1, 4], F[2, 5] = dt, dt, dt
        q = (process_noise_std ** 2)
        Q = np.diag([q*(dt**2), q*(dt**2), q*(dt**2), q, q, q])
        x_p = F @ x_filt[t-1]; P_p = F @ P_filt[t-1] @ F.T + Q
        x_pred[t], P_pred[t] = x_p, P_p
        z_t = y_pred[t]; y_k = z_t - H @ x_p
        S_k = H @ P_p @ H.T + R; K_k = P_p @ H.T @ inv(S_k)
        x_filt[t] = x_p + K_k @ y_k
        P_filt[t] = (np.eye(6) - K_k @ H) @ P_p
    x_smooth = np.zeros((N, 6)); P_smooth = np.zeros((N, 6, 6))
    x_smooth[-1], P_smooth[-1] = x_filt[-1], P_filt[-1]
    for t in range(N - 2, -1, -1):
        dt = max(0.01, float(delta_t_vec[t+1]))
        F = np.eye(6); F[0, 3], F[1, 4], F[2, 5] = dt, dt, dt
        C_k = P_filt[t] @ F.T @ inv(P_pred[t+1])
        x_smooth[t] = x_filt[t] + C_k @ (x_smooth[t+1] - x_pred[t+1])
        P_smooth[t] = P_filt[t] + C_k @ (P_smooth[t+1] - P_pred[t+1]) @ C_k.T
    return x_filt[:, :3], x_smooth[:, :3]

df_eval = pd.DataFrame({
    'user_id': uids, 'delta_t': dts,
    'pred_x': preds_raw[:, 0], 'pred_y': preds_raw[:, 1], 'pred_z': preds_raw[:, 2],
    'targ_x': targs[:, 0], 'targ_y': targs[:, 1], 'targ_z': targs[:, 2]
})

kf_preds = np.zeros_like(preds_raw)
rts_preds = np.zeros_like(preds_raw)

for uid in test_users:
    idx_u = df_eval['user_id'] == uid
    if not np.any(idx_u): continue
    udf = df_eval[idx_u]
    u_indices = udf.index.values
    y_u_true = udf[['targ_x', 'targ_y', 'targ_z']].values
    y_u_pred = udf[['pred_x', 'pred_y', 'pred_z']].values
    dt_u     = udf['delta_t'].values
    x_kf, x_rts = run_kalman_and_rts(y_u_true, y_u_pred, dt_u, process_noise_std=PROCESS_NOISE_STD, R_std=R_STD)
    kf_preds[u_indices]  = x_kf
    rts_preds[u_indices] = x_rts

mae_kf  = np.mean(np.linalg.norm(kf_preds - targs, axis=1))
mae_rts = np.mean(np.linalg.norm(rts_preds - targs, axis=1))
red_rts = ((mae_3d - mae_rts) / mae_3d) * 100.0

print('=== GRU + KINEMATIC SMOOTHING BENCHMARK (h=10) ===')
print(f'Raw GRU 3D MAE:        {mae_3d:.3f} meters')
print(f'Forward Linear KF:     {mae_kf:.3f} meters')
print(f'RTS Kalman Smoother:   {mae_rts:.3f} meters ({red_rts:+5.1f}%)')

## Trajectory Tracking Plots Across 5 Diverse Unseen Test Users

In [ ]:
# Select 5 diverse unseen test users for trajectory visualization
sorted_test_uids = sorted(list(test_users))
sample_5_uids = sorted_test_uids[:5]  # First 5 unseen test users

print(f'Generating and saving trajectory plots for 5 unseen test users: {sample_5_uids}')
print(f'Saving trajectory plots to: {PLOT_DIR}')

for u_idx, uid in enumerate(sample_5_uids, 1):
    idx_sample = df_eval['user_id'] == uid
    if not np.any(idx_sample): continue
    udf_s = df_eval[idx_sample]
    u_s_idx = udf_s.index.values
    
    gt_s  = udf_s[['targ_x', 'targ_y', 'targ_z']].values
    raw_s = preds_raw[u_s_idx]
    rts_s = rts_preds[u_s_idx]
    
    u_mae_raw = np.mean(np.linalg.norm(raw_s - gt_s, axis=1))
    u_mae_rts = np.mean(np.linalg.norm(rts_s - gt_s, axis=1))
    
    plt.figure(figsize=(10, 7.5))
    plt.plot(gt_s[:, 0], gt_s[:, 1], 'k-', lw=3.0, label=f'Ground Truth (User {uid})')
    plt.scatter(raw_s[:, 0], raw_s[:, 1], color='mediumpurple', alpha=0.55, s=22, label=f'Raw GRU (MAE: {u_mae_raw:.2f}m)')
    plt.plot(rts_s[:, 0], rts_s[:, 1], 'g-', lw=2.2, label=f'RTS Smoother (MAE: {u_mae_rts:.2f}m)')
    plt.xlabel('X Relative to Serving BS (meters)', fontsize=11)
    plt.ylabel('Y Relative to Serving BS (meters)', fontsize=11)
    plt.title(f'User {uid} Trajectory Tracking (GRU h=10 + RTS Smoother)', fontsize=12, fontweight='bold')
    plt.legend(fontsize=10, loc='upper right')
    plt.grid(True, alpha=0.3)
    
    plot_path = PLOT_DIR / f'trajectory_user_{uid}.png'
    plt.savefig(plot_path, dpi=150, bbox_inches='tight')
    plt.show()
    print(f'  [{u_idx}/5] Saved: {plot_path.name} | Raw MAE: {u_mae_raw:.2f}m | RTS MAE: {u_mae_rts:.2f}m')